https://colab.research.google.com/drive/1p-QYfws7TSdCez9xJNyvr2cwSMSSEGe3?usp=sharing

In [ ]:
# Install required packages
!pip install -q torch>=2.0.0
!pip install -q transformers>=4.30.0
!pip install -q datasets>=2.0.0
!pip install -q peft>=0.5.0
!pip install -q accelerate>=0.20.0
!pip install -q evaluate>=0.4.0
!pip install -q seqeval>=1.2.2
!pip install -q matplotlib>=3.7.0
!pip install -q scikit-learn>=1.3.0
!pip install -q gradio>=4.0.0
!pip install -q seaborn>=0.12.0

# Setup directory structure for MLOps
import os
from pathlib import Path

# Define paths
OUTPUT_DIR = Path("./medical_ner_finetuning")
PLOT_DIR = OUTPUT_DIR / "plots"
MODEL_DIR = OUTPUT_DIR / "model"
LOG_DIR = OUTPUT_DIR / "logs"

# Create directories
for d in [PLOT_DIR, MODEL_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Environment ready. Output directories created at: {OUTPUT_DIR}")

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import evaluate
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    TrainerCallback
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import confusion_matrix
import logging
import warnings
import re
from typing import List, Dict
import sys

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)
warnings.filterwarnings('ignore')

# Configuration and hyperparameters
CONFIG = {
    "model_name": "distilbert-base-uncased",
    "dataset_name": "AGBonnet/augmented-clinical-notes",
    "max_length": 512,
    "batch_size": 16,
    "lr": 2e-3,          # Increased learning rate for LoRA
    "epochs": 10,
    "lora_r": 32,        # LoRA rank
    "lora_alpha": 64,
    "lora_dropout": 0.1,
    "seed": 42
}

# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {device}")
if torch.cuda.is_available():
    logger.info(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Verify labels in training data
sample_idx = 0
sample_labels = train_ds[sample_idx]['labels']
sample_tokens = tokenizer.convert_ids_to_tokens(train_ds[sample_idx]['input_ids'])

print("Checking training sample 0...")
has_entity = False
for t, l in zip(sample_tokens, sample_labels):
    if l != -100 and l != 0:
        print(f"Token: {t} | Label ID: {l} ({id2label[l]})")
        has_entity = True

if not has_entity:
    print("WARNING: No entities found in this sample.")
else:
    print("SUCCESS: Labels are present.")

In [ ]:
# Weak supervision label generator with subword alignment

class MedicalNERLabelGenerator:
    """
    Implements weak supervision with robust subword alignment.
    Ensures that split words receive consistent labels across all subword parts.
    """
    def __init__(self):
        self.entity_patterns = {
            'CONDITION': [
                r'\b(hypertension|hypotension|diabetes|cancer|pneumonia|asthma|copd|failure|infarction|stroke|sepsis|fracture|pain|fever|anemia)\b',
                r'\b(myocardial infarction|multiple sclerosis|kidney disease|liver disease|heart failure)\b',
                r'\b(tachycardia|bradycardia|ischemia|stenosis|thrombosis|hemorrhage|edema|rash|diaphoresis)\b',
                r'\b(depression|anxiety|schizophrenia|bipolar|delirium)\b'
            ],
            'MEDICATION': [
                r'\b(aspirin|metformin|lisinopril|statin|insulin|warfarin|heparin|antibiotics)\b',
                r'\b(amoxicillin|ibuprofen|acetaminophen|morphine|fentanyl|prednisone|lasix)\b',
                r'\b(atorvastatin|simvastatin|omeprazole|metoprolol|albuterol|furosemide)\b'
            ],
            'PROCEDURE': [
                r'\b(surgery|biopsy|excision|amputation|transplant|intubation|dialysis)\b',
                r'\b(x-ray|ct scan|mri|ultrasound|ekg|ecg|colonoscopy|endoscopy|12-lead ekg)\b',
                r'\b(blood test|urinalysis|catheterization|stent placement|pci|cabg|intervention)\b'
            ],
            'ANATOMY': [
                r'\b(heart|lung|liver|kidney|brain|stomach|colon|bowel|bladder)\b',
                r'\b(chest|abdomen|head|neck|arm|leg|knee|spine|back|shoulder)\b',
                r'\b(artery|vein|nerve|muscle|bone|tissue|blood|left ventricle|leads)\b'
            ]
        }
        self.compiled_patterns = {
            label: [re.compile(p, re.IGNORECASE) for p in patterns]
            for label, patterns in self.entity_patterns.items()
        }

    def tokenize_and_label(self, text, tokenizer):
        # Tokenize input text
        encoding = tokenizer(text, truncation=True, max_length=CONFIG["max_length"],
                           padding='max_length', return_offsets_mapping=True)

        # Initialize all labels as 'O' (Outside entity)
        labels = [0] * len(encoding['input_ids'])

        # Label mapping for BIO tagging
        label_map = {
            'O': 0,
            'B-CONDITION': 1, 'I-CONDITION': 2,
            'B-MEDICATION': 3, 'I-MEDICATION': 4,
            'B-PROCEDURE': 5, 'I-PROCEDURE': 6,
            'B-ANATOMY': 7, 'I-ANATOMY': 8
        }

        offset_mapping = encoding['offset_mapping']

        # Apply weak supervision rules using regex patterns
        for entity_type, patterns in self.compiled_patterns.items():
            for pattern in patterns:
                for match in pattern.finditer(text):
                    start_char, end_char = match.span()

                    # Find all tokens within the matched entity span
                    match_tokens_indices = []
                    for idx, (token_start, token_end) in enumerate(offset_mapping):
                        if token_start is None or token_end is None or token_start == token_end:
                            continue

                        # Check for token overlap with entity
                        if token_start < end_char and token_end > start_char:
                            match_tokens_indices.append(idx)

                    # Assign BIO tags
                    if match_tokens_indices:
                        # First token gets B-tag
                        first_idx = match_tokens_indices[0]
                        labels[first_idx] = label_map[f'B-{entity_type}']

                        # Subsequent tokens get I-tag
                        for idx in match_tokens_indices[1:]:
                            labels[idx] = label_map[f'I-{entity_type}']

        return {
            'input_ids': encoding['input_ids'],
            'attention_mask': encoding['attention_mask'],
            'labels': labels
        }

In [ ]:
logger.info("Loading and processing data...")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
label_gen = MedicalNERLabelGenerator()

# Load dataset
try:
    raw_dataset = load_dataset(CONFIG["dataset_name"], split="train")

    # Find text column
    text_col = next((col for col in ['full_note', 'text', 'note'] if col in raw_dataset.column_names), None)
    if not text_col:
        raise KeyError(f"Could not find text column. Available: {raw_dataset.column_names}")

    # Filter for quality notes
    notes = [x[text_col] for x in raw_dataset if 50 < len(x[text_col].split()) < 1000]
    notes = notes[:1000]
    logger.info(f"Selected {len(notes)} quality notes for training.")

except Exception as e:
    logger.error(f"Dataset load failed: {e}")
    # Fallback data
    notes = ["Patient with severe hypertension and diabetes. Started on metformin."] * 100

# Generate labels using weak supervision
logger.info("Generating silver labels via weak supervision...")
tokenized_samples = [label_gen.tokenize_and_label(text, tokenizer) for text in notes]

# Create HuggingFace dataset
full_ds = Dataset.from_dict({
    'input_ids': [d['input_ids'] for d in tokenized_samples],
    'attention_mask': [d['attention_mask'] for d in tokenized_samples],
    'labels': [d['labels'] for d in tokenized_samples]
})

# Split into train and validation sets
dataset_split = full_ds.train_test_split(test_size=0.2, seed=CONFIG["seed"])
train_ds = dataset_split['train']
eval_ds = dataset_split['test']

logger.info(f"Data ready: {len(train_ds)} training, {len(eval_ds)} validation samples")

In [ ]:
# Verify labels in processed data
idx = 0
print(f"Checking Sample {idx} labels:")
tokens = tokenizer.convert_ids_to_tokens(full_ds[idx]['input_ids'])
labs = full_ds[idx]['labels']
for t, l in zip(tokens, labs):
    if l != 0 and l != -100:
        print(f"{t}: {id2label[l]}")

In [ ]:
# Model initialization with LoRA

# Label mappings for BIO tagging
id2label = {
    0: 'O',
    1: 'B-CONDITION', 2: 'I-CONDITION',
    3: 'B-MEDICATION', 4: 'I-MEDICATION',
    5: 'B-PROCEDURE', 6: 'I-PROCEDURE',
    7: 'B-ANATOMY', 8: 'I-ANATOMY'
}
label2id = {v: k for k, v in id2label.items()}

logger.info("Initializing base model...")
base_model = AutoModelForTokenClassification.from_pretrained(
    CONFIG["model_name"],
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id
)

# Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.TOKEN_CLS,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=["q_lin", "v_lin"],
    bias="none"
)

# Create PEFT model
model = get_peft_model(base_model, lora_config)

# Display trainable parameters
model.print_trainable_parameters()
logger.info("LoRA model ready.")

In [ ]:
# Training configuration

seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    """Computes precision, recall, F1, and accuracy using SeqEval."""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    # Filter out special tokens (-100)
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    eval_strategy="epoch",
    learning_rate=CONFIG["lr"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    num_train_epochs=CONFIG["epochs"],
    weight_decay=0.01,
    logging_dir=str(LOG_DIR),
    logging_steps=10,
    report_to="none",
    save_strategy="no"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics
)

logger.info("Trainer configured successfully.")

In [ ]:
# Training execution

logger.info("Starting training loop...")
train_result = trainer.train()

logger.info(f"Training complete. Time: {train_result.metrics['train_runtime']:.2f}s")

In [ ]:
# Visualization: learning curve and confusion matrix

# Plot learning curve
history = trainer.state.log_history
train_loss = [x['loss'] for x in history if 'loss' in x]
eval_loss = [x['eval_loss'] for x in history if 'eval_loss' in x]

plt.figure(figsize=(12, 5))

# Loss plot
plt.subplot(1, 2, 1)
plt.plot(train_loss, label='Training Loss', color='blue')
if eval_loss:
    steps = np.linspace(0, len(train_loss), len(eval_loss))
    plt.plot(steps, eval_loss, label='Validation Loss', color='orange', linestyle='--')
plt.title('Training Convergence (Loss)')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

# Confusion matrix
predictions, labels, _ = trainer.predict(eval_ds)
predictions = np.argmax(predictions, axis=2)

# Flatten and filter padding tokens
y_true = []
y_pred = []
for i in range(len(labels)):
    for j in range(len(labels[i])):
        if labels[i][j] != -100:
            y_true.append(id2label[labels[i][j]])
            y_pred.append(id2label[predictions[i][j]])

# Create and plot confusion matrix
labels_list = sorted(list(set(y_true)))
cm = confusion_matrix(y_true, y_pred, labels=labels_list)

plt.subplot(1, 2, 2)
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues',
            xticklabels=labels_list, yticklabels=labels_list)
plt.title('Entity Prediction Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')

plt.tight_layout()
plt.savefig(PLOT_DIR / "training_dashboard.png")
plt.show()

logger.info(f"Dashboard saved to {PLOT_DIR}")

In [ ]:
# Save model artifacts

logger.info("Saving model artifacts...")
model.save_pretrained(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))

print(f"Model saved to: {MODEL_DIR}")
print("   - adapter_model.bin (LoRA weights)")
print("   - adapter_config.json")
print("   - tokenizer.json")

In [ ]:
# Interactive Gradio demo
import gradio as gr
import torch.nn.functional as F

def analyze_note_polished(text):
    if not text:
        return "Please enter text."

    # Tokenize input
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)

    # Run inference
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    # Get predictions
    predictions = torch.argmax(outputs.logits, dim=2)[0].cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    # Merge subwords and extract entities
    entities = []
    curr_ent = []
    curr_type = None

    for token, pred in zip(tokens, predictions):
        if token in tokenizer.all_special_tokens:
            continue

        label = id2label[pred]

        if label.startswith("B-"):
            # Save previous entity
            if curr_ent:
                full_word = "".join(curr_ent).replace("##", "")
                entities.append((full_word, curr_type))

            # Start new entity
            curr_ent = [token]
            curr_type = label[2:]

        elif label.startswith("I-") and curr_type == label[2:]:
            # Continue current entity
            curr_ent.append(token)

        else:
            # Outside entity or type mismatch
            if curr_ent:
                full_word = "".join(curr_ent).replace("##", "")
                entities.append((full_word, curr_type))
            curr_ent = []
            curr_type = None

    # Save last entity
    if curr_ent:
        full_word = "".join(curr_ent).replace("##", "")
        entities.append((full_word, curr_type))

    # Format output
    output_str = "### Extracted Entities:\n"
    if not entities:
        output_str += "_No entities found._\n"
    else:
        # Group by type
        by_type = {}
        for ent, etype in entities:
            by_type.setdefault(etype, []).append(ent)

        for etype, ents in by_type.items():
            output_str += f"\n**{etype}:**\n"
            for e in ents:
                output_str += f"- {e}\n"

    return output_str

demo = gr.Interface(
    fn=analyze_note_polished,
    inputs=gr.Textbox(lines=5, label="Clinical Note"),
    outputs=gr.Markdown(label="Analysis Result"),
    title="Fine-Tuned Medical NER",
    description="DistilBERT + LoRA for medical entity extraction with subword merging.",
    examples=[
        ["Patient with hypertension and type 2 diabetes. Prescribed metformin."]
    ]
)

logger.info("Launching demo...")
demo.launch(share=True, debug=False)

In [ ]:
logger.info("Generating final presentation artifacts...")

# Training loss curve
history = trainer.state.log_history
train_loss = [x['loss'] for x in history if 'loss' in x]
steps = range(len(train_loss))

plt.figure(figsize=(10, 6))
plt.plot(steps, train_loss, label='Training Loss', color='#2ecc71', linewidth=2)
plt.title(f'LoRA Fine-Tuning Convergence\n(DistilBERT, r={CONFIG["lora_r"]}, lr={CONFIG["lr"]})', fontsize=14)
plt.xlabel('Training Steps', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / "final_learning_curve.png", dpi=300)
logger.info(f"Saved learning curve: {PLOT_DIR / 'final_learning_curve.png'}")

# Confusion matrix on evaluation set
logger.info("Computing confusion matrix...")
predictions, labels, _ = trainer.predict(eval_ds)
preds_flat = np.argmax(predictions, axis=2).flatten()
labels_flat = labels.flatten()

# Filter out special tokens and 'O' labels to focus on entities
mask = (labels_flat != -100) & (labels_flat != 0)
y_true = labels_flat[mask]
y_pred = preds_flat[mask]

# Map IDs to label names
target_names = [id2label[i] for i in sorted(list(set(y_true)))]
y_true_names = [id2label[i] for i in y_true]
y_pred_names = [id2label[i] for i in y_pred]

if len(target_names) > 0:
    plt.figure(figsize=(10, 8))
    cm = confusion_matrix(y_true_names, y_pred_names, labels=target_names)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=target_names, yticklabels=target_names)
    plt.title('Entity Classification Confusion Matrix', fontsize=14)
    plt.ylabel('Actual Label')
    plt.xlabel('Predicted Label')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "confusion_matrix.png", dpi=300)
    logger.info(f"Saved confusion matrix: {PLOT_DIR / 'confusion_matrix.png'}")
else:
    logger.warning("Not enough entity predictions to generate confusion matrix.")

print("\n" + "="*60)
print("PROJECT COMPLETE: READY FOR PRESENTATION")
print("="*60)
print(f"1. Model Artifacts: {MODEL_DIR}")
print(f"2. Visualizations:  {PLOT_DIR}")
print("3. Demo:            Active (in Cell 10)")